# Sequence Creation for LSTM

## Imports and Data Loading

In [14]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
 
print("Imports successful")
 
# Load raw data from feature engineering
print("\nLoading feature data from feature-engineering/...")
X_train = np.load('../../feature-engineering/X_train.npy')
X_test = np.load('../../feature-engineering/X_test.npy')
y_train = np.load('../../feature-engineering/y_train.npy')
y_test = np.load('../../feature-engineering/y_test.npy')
 
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
 
# Load feature names
with open('../../feature-engineering/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)
 
with open('../../feature-engineering/pitch_type_encoder.pkl', 'rb') as f:
    pitch_encoder = pickle.load(f)
 
print(f"\nLoaded {len(feature_names)} features and {len(pitch_encoder.classes_)} pitch types.")

Imports successful

Loading feature data from feature-engineering/...
X_train shape: (482656, 26)
X_test shape: (117758, 26)
y_train shape: (482656,)
y_test shape: (117758,)

Loaded 26 features and 17 pitch types.


## Load original DF to reconstruct ABs

In [15]:
df_all = pd.read_csv('../../EDA/mlb_pitches_eda.csv')
print(f"Loaded {len(df_all)} original pitch records")
 
# Create AB identifier (game_date + pitcher + batter)
df_all['ab_id'] = (
    df_all['game_date'].astype(str) + '_' +
    df_all['pitcher'].fillna(-1).astype(int).astype(str) + '_' +
    df_all['batter'].fillna(-1).astype(int).astype(str)
)
 
print(f"Unique ABs: {df_all['ab_id'].nunique()}")

Loaded 600414 original pitch records
Unique ABs: 98269


## Sequence Creation

In [16]:
SEQ_LENGTH = 5
 
def create_lstm_sequences(X, y, df, seq_length=5):
    sequences_X = []
    sequences_y = []

    # Group by AB
    for ab_id, group_indices in df.groupby('ab_id').groups.items():
        group_indices = sorted(group_indices)
        
        if len(group_indices) < seq_length + 1:
            # AB too short; need at least seq_length+1 pitches
            continue
        
        # Create sliding windows for this AB
        for i in range(len(group_indices) - seq_length):
            # Input: features of last seq_length pitches
            seq_indices = group_indices[i:i+seq_length]
            seq_features = X[seq_indices]  # Shape: (seq_length, n_features)
            
            # Target: pitch type of next pitch (pitch at i+seq_length)
            target_idx = group_indices[i + seq_length]
            target_pitch = y[target_idx]
            
            sequences_X.append(seq_features)
            sequences_y.append(target_pitch)
    
    X_seq = np.array(sequences_X)  # Shape: (n_sequences, seq_length, n_features)
    y_seq = np.array(sequences_y)  # Shape: (n_sequences)
    
    return X_seq, y_seq
 
print(f"Creating sequences with seq_length={SEQ_LENGTH}...")
 
# We need to match indices from raw data to X_train/X_test. 
# We'll create sequences directly from the available X/y data and group by pitches that came from the same AB.
 
# Filter to only pitchers in train/test sets
print("\nLoading test pitcher IDs...")
test_pitcher_ids = np.load('../../feature-engineering/test_pitcher_ids.npy')
train_pitcher_ids = np.array([p for p in df_all['pitcher'].unique() if p not in test_pitcher_ids])
 
print(f"Train pitchers: {len(train_pitcher_ids)}")
print(f"Test pitchers: {len(test_pitcher_ids)}")
 
# Sort and filter dataframe
df_sorted = df_all.sort_values(['game_date', 'pitcher', 'batter', 'pitch_number']).reset_index(drop=True)
 
df_train = df_sorted[df_sorted['pitcher'].isin(train_pitcher_ids)].reset_index(drop=True)
df_test = df_sorted[df_sorted['pitcher'].isin(test_pitcher_ids)].reset_index(drop=True)
 
print(f"Train pitches in df: {len(df_train)}")
print(f"Test pitches in df: {len(df_test)}")
print(f"Expected train pitches: {len(X_train)}")
print(f"Expected test pitches: {len(X_test)}")
 
# Create sequences using the filtered dataframes
X_train_seq, y_train_seq = create_lstm_sequences(X_train, y_train, df_train, seq_length=SEQ_LENGTH)
X_test_seq, y_test_seq = create_lstm_sequences(X_test, y_test, df_test, seq_length=SEQ_LENGTH)
 
print(f"\nTraining sequences: {X_train_seq.shape}")
print(f"  Input shape: (n_sequences={X_train_seq.shape[0]}, seq_length={X_train_seq.shape[1]}, n_features={X_train_seq.shape[2]})")
print(f"  Target shape: {y_train_seq.shape}")
 
print(f"\nTest sequences: {X_test_seq.shape}")
print(f"  Input shape: (n_sequences={X_test_seq.shape[0]}, seq_length={X_test_seq.shape[1]}, n_features={X_test_seq.shape[2]})")
print(f"  Target shape: {y_test_seq.shape}")

Creating sequences with seq_length=5...

Loading test pitcher IDs...
Train pitchers: 665
Test pitchers: 117758
Train pitches in df: 482656
Test pitches in df: 117758
Expected train pitches: 482656
Expected test pitches: 117758

Training sequences: (160550, 5, 26)
  Input shape: (n_sequences=160550, seq_length=5, n_features=26)
  Target shape: (160550,)

Test sequences: (38056, 5, 26)
  Input shape: (n_sequences=38056, seq_length=5, n_features=26)
  Target shape: (38056,)


## Saving the Sequences

In [17]:
np.save('X_train_seq.npy', X_train_seq)
np.save('X_test_seq.npy', X_test_seq)
np.save('y_train_seq.npy', y_train_seq)
np.save('y_test_seq.npy', y_test_seq)
 
print("Saved sequence arrays:")
print(f"  X_train_seq.npy ({X_train_seq.shape})")
print(f"  X_test_seq.npy ({X_test_seq.shape})")
print(f"  y_train_seq.npy ({y_train_seq.shape})")
print(f"  y_test_seq.npy ({y_test_seq.shape})")
 
# Save metadata
metadata = {
    'seq_length': SEQ_LENGTH,
    'n_features': X_train_seq.shape[2],
    'n_pitch_types': len(pitch_encoder.classes_),
    'pitch_encoder': pitch_encoder,
    'feature_names': feature_names,
}
 
with open('seq_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
 
print("\nSaved seq_metadata.pkl")

Saved sequence arrays:
  X_train_seq.npy ((160550, 5, 26))
  X_test_seq.npy ((38056, 5, 26))
  y_train_seq.npy ((160550,))
  y_test_seq.npy ((38056,))

Saved seq_metadata.pkl
